# 🧠 PRD-LLM Backend (Colab GPU Runner)

ဒီ Notebook က PRD-LLM ရဲ့ ဉာဏ်ရည်တု အင်ဂျင်ကို Colab GPU သုံးပြီး run ပေးမှာ ဖြစ်ပါတယ်။

### 🛠 အရေးကြီးသော Setup (မrunမီ ဒါကို အရင်လုပ်ပါ)
1. ဘယ်ဘက်ဘေးက **Key icon (Secrets)** ကို နှိပ်ပါ။
2. **Add new secret** ကို နှိပ်ပြီး `NGROK_AUTH_TOKEN` ဆိုတဲ့ နာမည်နဲ့ Token ထည့်ပါ။
3. အဲဒီ Key ရဲ့ ဘေးက **Notebook access ခလုတ်ကို အပြာရောင်ဖြစ်အောင် ဖွင့်ပေးပါ** (ဒါမှ code က key ကို ယူသုံးလို့ရမှာပါ)။
4. **သတိပြုရန်:** အပေါ်ဆုံးမှာ တက်နေတဲ့ `userdata.get('secretName')` ဆိုတဲ့ cell အမှားကို run စရာမလိုပါ။ ဖျက်ပစ်လိုက်ပါ။

In [ ]:
# [အဆင့် ၁] - လိုအပ်သော Library များ Install လုပ်ခြင်း
print("📦 Installing essential libraries...")
!pip install fastapi uvicorn pyngrok nest-asyncio python-dotenv google-generativeai
print("✅ Installation Complete!")

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
pyngrok
nest-asyncio
python-dotenv
google-generativeai

In [ ]:
%%writefile COLAB_SERVER.py
import os
import uvicorn
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio

app = FastAPI(title="PRD-LLM Relay Server")

@app.get("/")
async def root():
    return {"status": "active", "engine": "PRD-LLM"}

@app.get("/api/health")
async def health():
    return {"status": "ok"}

def start_server():
    nest_asyncio.apply()
    
    # Ngrok configuration
    token = os.environ.get("NGROK_AUTH_TOKEN")
    if token:
        ngrok.set_auth_token(token)
        public_url = ngrok.connect(8000).public_url
        print(f"\n🚀 PRD-LLM RELAY IS ACTIVE!")
        print(f"🔗 RELAY URL: {public_url}\n")
    else:
        print("⚠️ Warning: NGROK_AUTH_TOKEN not found in environment.")

    uvicorn.run(app, host="0.0.0.0", port=8000)

if __name__ == "__main__":
    start_server()

In [ ]:
# [အဆင့် ၂] - Server စတင်ခြင်း
from google.colab import userdata
import os

print("🔍 Checking Secrets...")
try:
    # Secrets ထဲက key ကို ဖတ်မယ်
    token = userdata.get('NGROK_AUTH_TOKEN')
    if not token:
        # အကယ်၍ Token_1 လို့ ပေးထားခဲ့ရင်လည်း စစ်ပေးမယ်
        try: token = userdata.get('NGROK_AUTH_TOKEN_1')
        except: pass
        
    if not token:
        raise ValueError("NGROK_AUTH_TOKEN ကို Secrets (Key icon) ထဲမှာ ရှာမတွေ့ပါ။ Notebook access ဖွင့်ထားဖို့ မမေ့ပါနဲ့။")
    
    os.environ['NGROK_AUTH_TOKEN'] = token
    print("✅ Ngrok Token successfully loaded.")
    
    # Server Start
    print("🚀 Starting PRD-LLM Engine...")
    !python COLAB_SERVER.py
except Exception as e:
    print("❌ ERROR: Setup မပြည့်စုံပါ။")
    print(f"အသေးစိတ်: {e}")